# Customer Exploratory Data Analysis

This notebook explores customer behavioral features to understand their
distributions, identify potential outliers, examine relationships between
features, and evaluate their suitability for customer segmentation.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

## 2. Load Customer Dataset

In [ ]:
sql_path = Path("..") / "01_data_preparation" / "01_customer_dataset.sql"

In [ ]:
server = os.getenv("SQL_SERVER", "localhost")
database = "EcommerceAnalysis"

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

encoded_connection_string = quote_plus(connection_string)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={encoded_connection_string}"
)

In [ ]:
customer_query = sql_path.read_text(encoding="utf-8")

In [ ]:
df = pd.read_sql(customer_query, engine)

## 3. Feature Distribution Analysis

Analyze the distributions of customer behavioral features to identify
skewness, concentration, and potential extreme values.

In [ ]:
sns.histplot(data=df, x="recency_days", bins=30)

plt.title("Distribution of Customer Recency")
plt.xlabel("Recency (Days)")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
sns.histplot(data=df, x="frequency", bins=30)

plt.title("Distribution of Customer Frequency")
plt.xlabel("Number of Delivered Orders")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
sns.histplot(data=df, x="monetary", bins=30)

plt.title("Distribution of Customer Monetary Value")
plt.xlabel("Total Customer Spend")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
sns.boxplot(data=df, x="monetary")

plt.title("Boxplot of Customer Monetary Value")
plt.xlabel("Total Customer Spend")

plt.show()

In [ ]:
q1 = df["monetary"].quantile(0.25)
q3 = df["monetary"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

In [ ]:
(df["monetary"] > upper_bound).sum()

In [ ]:
outlier_count = (df["monetary"] > upper_bound).sum()
total_customers = len(df)
outlier_percentage = outlier_count * 100 / total_customers
outlier_percentage

In [ ]:
sns.histplot(data=df, x="avg_order_value", bins=30)

plt.title("Distribution of Average Order Value")
plt.xlabel("Average Order Value")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
sns.histplot(data=df, x="purchase_frequency", bins=30)

plt.title("Distribution of Customer Purchase Frequency")
plt.xlabel("Monthly Purchase Frequency")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
sns.histplot(data=df, x="returned_order_rate", bins=30)

plt.title("Distribution of Returned Order Rate")
plt.xlabel("Returned Order Rate")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
sns.histplot(data=df, x="average_rating", bins=30)

plt.title("Distribution of Customer Average Rating")
plt.xlabel("Average Rating")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
sns.histplot(data=df, x="review_rate", bins=30)

plt.title("Distribution of Customer Review Rate")
plt.xlabel("Review Rate")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
sns.histplot(data=df, x="effective_discount_rate", bins=30)

plt.title("Distribution of Effective Discount Rate")
plt.xlabel("Effective Discount Rate")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
sns.histplot(data=df, x="avg_items_per_order", bins=30)

plt.title("Distribution of Average Items per Order")
plt.xlabel("Average Items per Order")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
sns.histplot(data=df, x="category_diversity", bins=30)

plt.title("Distribution of Customer Category Diversity")
plt.xlabel("Customer Category Diversity")
plt.ylabel("Number of Customers")

plt.show()

### Key Findings

- Recency, frequency, monetary value, average order value, and purchase frequency show right-skewed distributions, indicating that most customers are concentrated at lower behavioral values while a smaller group exhibits substantially higher activity or value.
- Monetary value contains notable high-value observations. Using the IQR method, approximately 6.63% of customers were identified as potential upper outliers. These observations will be retained for further analysis rather than removed automatically.
- Returned order rate is strongly concentrated around zero, indicating that most customers have no returned orders associated with their delivered purchases.
- Average customer ratings are generally concentrated between 3 and 4+, while missing ratings represent customers without rating information and should not be interpreted as zero ratings.
- Effective discount rate and average items per order show relatively balanced distributions compared with the strongly right-skewed purchasing features.
- Category diversity is concentrated at lower-to-moderate values, while a smaller group of customers purchases across a wider range of product categories.

## 4. Feature Relationships and Correlation

Analyze relationships between customer behavioral features to identify
patterns, dependencies, and potentially redundant information before
customer segmentation.

In [ ]:
sns.scatterplot(data=df, x='frequency', y='monetary')

plt.title('Frequency vs Monetary Value')
plt.xlabel('Number of Delivered Orders')
plt.ylabel('Total Customer Spend')

plt.show()

In [ ]:
df['frequency'].corr(df['monetary'])

In [ ]:
sns.scatterplot(data=df, x='frequency', y='purchase_frequency')

plt.title('Frequency vs Purchase Frequency')
plt.xlabel('Number of Delivered Order')
plt.ylabel('Monthly Purchase Frequency')

plt.show()

In [ ]:
df['frequency'].corr(df['purchase_frequency'])

In [ ]:
sns.scatterplot(data=df, x="monetary", y="avg_order_value")

plt.title("Monetary vs Average Order Value")
plt.xlabel("Total Customer Spend")
plt.ylabel("Average Order Value")

plt.show()

In [ ]:
df["monetary"].corr(df["avg_order_value"])

In [ ]:
numeric_features = df.select_dtypes(include=np.number)

In [ ]:
numeric_features.columns

In [ ]:
correlation_matrix = numeric_features.corr()

In [ ]:
plt.figure(figsize=(12, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f"
)

plt.title("Correlation Matrix of Customer Features")

plt.show()

In [ ]:
upper_mask = np.triu(
    np.ones(correlation_matrix.shape),
    k=1
)

In [ ]:
upper_corr = correlation_matrix.where(upper_mask == 1)

In [ ]:
upper_corr

In [ ]:
strong_correlations = upper_corr[
    upper_corr.abs() >= 0.7
]

In [ ]:
upper_corr.abs()

In [ ]:
strong_correlations

In [ ]:
numeric_features.skew()

In [ ]:
numeric_features.agg(["min", "max"])

## 5. Feature Selection Considerations

Based on the exploratory analysis, the following considerations will guide
feature engineering and customer segmentation:

- `recency_days`, `frequency`, and `monetary` represent the core RFM dimensions
  and will be retained as key behavioral features.

- `monetary`, `avg_order_value`, and `purchase_frequency` show strong positive
  skewness and will be considered for log transformation before clustering.

- `avg_items_per_order` provides additional information about customer basket
  size and will be retained.

- `category_diversity` captures the breadth of customer purchasing behavior.
  Although it is highly correlated with `frequency`, it provides distinct
  behavioral meaning and will be retained for further evaluation.

- `favorite_category_share` captures category concentration. Its strong negative
  correlation with `category_diversity` indicates potential redundancy, so its
  contribution will be evaluated during clustering experiments.

- `effective_discount_rate` captures customers' effective discount exposure and
  will be retained as a candidate segmentation feature.

- `returned_order_rate` is strongly zero-heavy. It contains potentially useful
  return behavior information but will be evaluated carefully during clustering.

- `average_rating` and `review_rate` will not be included in the primary
  clustering feature set. They will instead be retained for customer and cluster
  profiling.

- Large differences in feature scales indicate that feature scaling will be
  required before applying distance-based clustering algorithms such as K-Means.

### Candidate Clustering Features

- recency_days
- frequency
- monetary
- avg_order_value
- avg_items_per_order
- category_diversity
- favorite_category_share
- purchase_frequency
- effective_discount_rate
- returned_order_rate